In [ ]:
import sys; sys.path.append('..'); sys.path.append('../../../');  sys.path.append('../../../gmsh')

In [ ]:
import experiment_helper
import igl
from periodic_simulation_setup import *
import json


In [ ]:
angle = 0
r = 2

In [ ]:
import importlib
importlib.reload(pattern_generator_using_gmsh)

In [ ]:
a = 2
avg_len = 0.05
ipu, m, marker = pattern_generator_using_gmsh.get_three_star_hex(a, avavg_len, avg_len)        

In [ ]:
visualization.plot_2d_mesh(m, pointList = marker, width = 20, height = 20)

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)


In [ ]:
viewer.showWireframe(False)

In [ ]:
viewer.show()

In [ ]:
# pressure = 0.8
stiffness_pressure = 0.4
scale_factor_pressure = 0.01

In [ ]:
allowBending = False
useTFT = True
disableFusedRegionTFT = False

In [ ]:
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
def cb(i):
    viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
ipu.visualizationTilePower = 0

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

benchmark.reset()
print(allowBending, stiffness_pressure, hessianShift, fixedVars)

opts.niter = 500
opts.gradTol = 1e-10
print(opts.factorizer)

cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
benchmark.report()

In [ ]:
curr_vars = ipu.getVars()
curr_vars[-1] =np.pi / 2
curr_vars[-2] = -0.3
ipu.setVars(curr_vars)

In [ ]:
az_ipu = get_az_ipu_from_ipu(ipu, m, marker, useTFT, disableFusedRegionTFT)
if not allowBending:
    fixedVars, hessianShift = [az_ipu.numVars() - 2, az_ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6
opts.niter = 1000
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=None, hessianShift = hessianShift)
# cr = az_optimizer.optimize()

In [ ]:
plt.plot(stiffness_values)

In [ ]:
plt.plot(stiffness_values)

In [ ]:
points = visualize_average_deformation_gradient(ipu, 100)


In [ ]:

name = "three_star_hex"
time_stamp = time.strftime("%Y_%m_%d_%H_%M")
result_folder = 'output/{}/{}'.format(name, time_stamp)
if not os.path.exists(result_folder):
    os.makedirs(result_folder)  


In [ ]:
variable = 0
low_pressure_tag = "low_pressure"
high_pressure_tag = "high_pressure"


In [ ]:
np.save("{}/stiffness_values_{}_{}.npy".format(result_folder, name, variable), stiffness_values)
np.save("{}/sampled_alphas_{}_{}.npy".format(result_folder, name, variable), sampled_alphas)
print("Solved using stiffness shift: ", stiffness_shift)

In [ ]:

np.save("{}/experiment_parameters.npy".format(result_folder), np.array([stiffness_pressure, scale_factor_pressure, allowBending, disableFusedRegionTFT, useTFT]))

np.save("{}/{}_strain_values_{}_{}.npy".format(result_folder, high_pressure_tag, name, variable), utils.getStrains(az_ipu.ipu.sheet)[:, 0])


render = viewer.offscreenRenderer(1000, 1000)
render.render()
render.save("{}/{}_render_{}_{}.png".format(result_folder, high_pressure_tag, name, variable))
np.save("{}/{}_dofs_{}_{}.npy".format(result_folder, high_pressure_tag, name, variable), az_ipu.getVars())

In [ ]:
np.save("{}/scale_factors_{}_{}.npy".format(result_folder, name, variable), get_deformation_scale_factors(az_ipu.ipu))
np.save("{}/kappa_{}_{}.npy".format(result_folder, name, variable), az_ipu.getVars()[-2])

np.save("{}/average_deformation_gradient_matrix_{}_{}.npy".format(result_folder, name, variable), get_deformation_matrix(az_ipu.ipu))

np.save("{}/{}_strain_values_{}_{}.npy".format(result_folder, low_pressure_tag, name, variable), utils.getStrains(az_ipu.ipu.sheet)[:, 0])

np.save("{}/{}_dofs_{}_{}.npy".format(result_folder, low_pressure_tag, name, variable), az_ipu.getVars())

In [ ]:
import periodic_simulation_setup
importlib.reload(periodic_simulation_setup)
from periodic_simulation_setup import *

In [ ]:
stiffness_shift = 1e-15
success = False
for i in range(15):
    try:
        stiffness_values, sampled_alphas = visualize_sampled_bending_stiffness(az_ipu, 1000, az_optimizer, hessianShift = stiffness_shift, fixedVars = [], filename = "{}/stiffness_{}_{}.png".format(result_folder, name, variable), plot_min_r=0)
        np.save("{}/stiffness_values_{}_{}.npy".format(result_folder, name, variable), stiffness_values)
        np.save("{}/sampled_alphas_{}_{}.npy".format(result_folder, name, variable), sampled_alphas)
        print("Solved using stiffness shift: ", stiffness_shift)
        success = True
        break
    except:
        print("failed to compute stiffness with shift ", stiffness_shift)
        stiffness_shift *= 10
if (not success):
    print("Failed to solve stiffness!")



In [ ]:
max(az_ipu.ipu.sheet.getVars().reshape((int(az_ipu.ipu.sheet.numVars() / 3), 3))[:, 2])

In [ ]:
points = visualize_average_deformation_gradient(ipu, 100, filename = "{}/average_deformation_gradient_{}_{}.png".format(result_folder, name, variable), plot_min_r=0, plot_max_r=1)
